In [ ]:
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

df = pd.read_csv('marketing_campaign.csv')
df.columns = df.columns.str.strip().str.replace(' ', '_')

df_income = df.dropna(subset=['Income', 'Response']).copy()

df_income['Response'] = df_income['Response'].astype(str)
df_income['Response_Label'] = df_income['Response'].map({'1': 'Accepted Offer (n={})'.format(len(df_income[df_income['Response']=='1'])),
                                                     '0': 'Rejected Offer (n={})'.format(len(df_income[df_income['Response']=='0']))})

income_accept = df_income[df_income['Response']=='1']['Income']
income_reject = df_income[df_income['Response']=='0']['Income']

t_stat, p_val = stats.ttest_ind(income_accept, income_reject, equal_var=False)

print("T-Test: Income vs Campaign Response (Offer Acceptance)")
print("-" * 50)
print(f"Accepted Offer Mean Income: ${income_accept.mean():,.2f}, n={len(income_accept)}")
print(f"Rejected Offer Mean Income: ${income_reject.mean():,.2f}, n={len(income_reject)}")
print(f"T-Statistic (Welch's): {t_stat:.4f}")
print(f"P-Value: {p_val:.4f}")
if p_val < 0.05:
    print("Conclusion: The difference in mean income between the two groups IS statistically significant (p < 0.05).")
else:
    print("Conclusion: The difference in mean income between the two groups is NOT statistically significant (p >= 0.05).")
print("-" * 50)

plt.figure(figsize=(10, 6))

sns.violinplot(
    x='Response_Label', 
    y='Income', 
    data=df_income, 
    hue='Response_Label',
    palette=['#FF5733', '#33FF57'],
    inner='quartile', 
    linewidth=1.5,
    legend=False
)

plt.title(
    "Comparison of Customer Income by Campaign Response", 
    fontsize=16, 
    fontweight='bold'
)
plt.xlabel("Campaign Response Group", fontsize=12)
plt.ylabel("Annual Income (USD)", fontsize=12)
plt.yticks(rotation=0)

plt.text(
    0.5, 
    df_income['Income'].max() * 0.95,
    f"Welch's T-Test Results:\nT-Stat: {t_stat:.2f}\nP-Value: {p_val:.4f}",
    horizontalalignment='center',
    bbox={'facecolor': 'lightgray', 'alpha': 0.7, 'pad': 5}
)

formatter = ticker.FuncFormatter(lambda x, p: '${:,.0f}'.format(x))
plt.gca().yaxis.set_major_formatter(formatter)

plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()